# 1. O Minimundo

O projeto consiste na modelagem e implementação do banco de dados relacional para um **Sistema de Gestão de Biblioteca Escolar**. O objetivo principal é automatizar o controle de acervo físico, com opções de empréstimos e devoluções, gestão de usuários, equipe de atendimento, fila de espera e arrecadação de multas por atraso.

### Detalhamento das Regras de Negócio e Casos de Uso:

1. Gestão do Acervo e Catalogação:
   - Cada **Livro** é identificado univocamente pelo seu `ISBN` e associado a uma **Editora** e a uma **Categoria**.
   - A hierarquia de **Categorias** é recursiva, no caso, ela permite subcategorias, mas possui validação estrita para impedir ciclos infinitos.
   - Os **Autores** possuem nacionalidade e são vinculados aos livros com a indicação da `ordem_autoria`, sendo essa, por ex: 1º autor, 2º autor.

2. Distinção de Exemplares Físicos:
   - O sistema **não empresta o livro de forma abstrata**, mas sim uma **cópia física específica**, o chamado `Exemplar`, identificada pela chave composta `(ISBN, Número do Exemplar)`.
   - Cada exemplar possui estado de conservação, registrado por `NOVO`, `BOM`, `REGULAR`, `RUIM`, `EM_REPARO`, e localização física da estante na biblioteca.

3. Empréstimos e Devoluções:
   - O registro de **Empréstimo** vincula um Usuário a um Exemplar específico, registrando a data da retirada, a data prevista para devolução e a taxa diária de multa negociada no momento, indicado por `multa_diaria`.
   - Cada exemplar só pode ter **um empréstimo em aberto por vez**, garantido via índice único parcial no PostgreSQL.
   - O sistema armazena a rastreabilidade do **Funcionário** que realizou a entrega e do funcionário que recebeu a devolução.

4. **Fila de Reservas FIFO:**
   - Quando todas as cópias físicas de um título estiverem emprestadas, os usuários podem registrar uma **Reserva**.
   - A fila funciona estritamente por ordem cronológica de solicitação, seguindo a lógica *First-In, First-Out*.

5. **Multas e Pagamentos:**
   - Os dias de atraso são calculados comparando a data prevista com a data de devolução efetiva, ou a data atual para empréstimos em aberto.
   - O valor devido é abatido conforme o histórico de registros na tabela `PAGAMENTO_MULTA`, permitindo quitação parcial ou total via PIX, dinheiro ou cartão.

# 2. O Diagrama Entidade-Relacionamento (DER) -> Primeira Versão

A versão inicial do modelo focava em uma estrutura relacional simplificada, contendo as tabelas Aluno, Livro, Editora, Autor, Emprestimo e Livro_Autor.

### Análise das Limitações:

- Após análise mais profunda a respeito do diagrama, percebemos que ele estava muito simples e possuiam algumas limitações. Observamos que o empréstimo relacionava a tabela `Aluno` diretamente com a tabela `Livro` via `ISBN`. Caso a biblioteca possuísse 3 exemplares de um mesmo livro, o banco não conseguia controlar qual tomo físico específico estava com qual leitor. Além disso, a tabela de empréstimo continha apenas `Data_Inicio` e `Data_Fim` e não havia registro do prazo limite acordado, impossibilitando o cálculo de atrasos e multas. Outro fator é que os dados do leitor eram superficiais, não existia separação de endereço, e-mail único ou suporte para mais de um telefone. Também notamos que o sistema ignorava os funcionários que atendiam o leitor, o cadastro de categorias do acervo, a fila de espera de livros esgotados e a baixa de pagamentos.

# 3. Evolução Arquitetural -> Comparativo DER v1 x DER v2

A segunda versão reestruturou o banco para suportar todas as operações de uma biblioteca e corrigir as falhas da primeira versão.

### Tabela de Mapeamento das Mudanças e Justificativas de Engenharia:

| Componente / Regra | Primeira Versão | Segunda versão | Justificativa|
| :--- | :--- | :--- | :--- |
| **Identificação do Leitor** | Entidade `Aluno`| Entidade `USUARIO` com `Matrícula`, `CPF`, endereço fragmentado e `USUARIO_TELEFONE` | A Matrícula passa a ser a chave primária estável. O CPF único e validador previne cadastros duplicados. |
| **Unidade de Empréstimo** | Relacionava `Aluno` direto com `Livro` via ISBN | Criação da entidade `EXEMPLAR` com chave primária composta `(isbn, numero_exemplar)` | Permite gerenciar múltiplos exemplares físicos do mesmo livro, controlar a conservação individual e impedir que 2 leitores retirem o mesmo livro ao mesmo tempo. |
| **Histórico de Circulação** | Apenas `Data_Inicio` e `Data_Fim` | Adicionados `data_emprestimo`, `data_prevista`, `data_devolucao` e `multa_diaria` | Permite o cálculo dinâmico do atraso, feito por `dias_atraso = data_devolucao - data_prevista`, e congela o valor cobrado por dia na data do contrato de empréstimo. |
| **Organização do Acervo** | Atributo simples ou inexistente | Entidade `CATEGORIA` com auto-relacionamento | Permite criar árvores de tópicos.|
| **Responsabilidade Operacional** | Não existia | Entidade `FUNCIONARIO` com auto-relacionamento de supervisão | Rastreia qual funcionário liberou o livro, qual recebeu na devolução e qual deu baixa na multa. Permite hierarquia entre bibliotecários e supervisores. |
| **Fila de Espera** | Não existia | Entidade `RESERVA` com status `ATIVA`, `ATENDIDA` ou `CANCELADA` | Gerencia a fila por ordem cronológica quando todos os exemplares de um título estão emprestados. |
| **Caixa e Finanças** | Não existia | Entidade `PAGAMENTO_MULTA` com forma de pagamento | Permite dar baixa gradual no saldo devedor de multas, suportando pagamentos parciais e gerando histórico financeiro. |

---
# 4. Instalação e Configuração da Infraestrutura

In [1]:
# instalação do postgresql na máquina do colab
!sudo apt update -qq
!sudo apt install -y postgresql postgresql-contrib
!sudo service postgresql start
!sudo -u postgres createdb biblioteca

# comando bash para digitar menos: posql -c "comando sql"
!echo '#!/bin/bash' > /usr/local/bin/posql
!echo 'sudo -u postgres psql "$@" -d biblioteca & sudo -u postgres psql -c "\q"' >> /usr/local/bin/posql
!chmod +x /usr/local/bin/posql

!posql -c "select current_database();"

10 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libcommon-sense-perl libjson-perl libjson-xs-perl libtypes-serialiser-perl
  logrotate postgresql-16 postgresql-client-16 postgresql-client-common
  postgresql-common ssl-cert sysstat
Suggested packages:
  bsd-mailx | mailx postgresql-doc postgresql-doc-16 isag
The following NEW packages will be installed:
  libcommon-sense-perl libjson-perl libjson-xs-perl libtypes-serialiser-perl
  logrotate postgresql postgresql-16 postgresql-client-16
  postgresql-client-common postgresql-common postgresql-contrib ssl-cert
  sysstat
0 upgraded, 13 newly installed, 0 to remove and 10 not upgraded

---
# 5. Construção da Estrutura Físico-Relacional (DDL)
Criação das tabelas, chaves primárias/estrangeiras, validações lógicas via `CHECK`, regras de integridade e visões (`VIEWS`) para consultas simplificadas.

In [2]:
%%bash
posql -c "
BEGIN;

CREATE TABLE IF NOT EXISTS configuracao_biblioteca (
    id SMALLINT PRIMARY KEY CHECK (id = 1),
    dias_emprestimo_padrao SMALLINT NOT NULL CHECK (dias_emprestimo_padrao > 0),
    multa_diaria NUMERIC(8,2) NOT NULL CHECK (multa_diaria >= 0),
    atualizado_em TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO configuracao_biblioteca (id, dias_emprestimo_padrao, multa_diaria)
VALUES (1, 14, 1.00)
ON CONFLICT (id) DO NOTHING;

CREATE TABLE IF NOT EXISTS editora (
    id_editora SERIAL PRIMARY KEY,
    nome VARCHAR(160) NOT NULL UNIQUE,
    cidade VARCHAR(100),
    site VARCHAR(240)
);

CREATE TABLE IF NOT EXISTS categoria (
    id_categoria SERIAL PRIMARY KEY,
    nome VARCHAR(100) NOT NULL UNIQUE,
    id_categoria_pai INTEGER REFERENCES categoria(id_categoria) ON DELETE SET NULL,
    CONSTRAINT ck_categoria_nao_pode_ser_pai_de_si CHECK (id_categoria_pai IS DISTINCT FROM id_categoria)
);

CREATE TABLE IF NOT EXISTS livro (
    isbn VARCHAR(20) PRIMARY KEY,
    titulo VARCHAR(240) NOT NULL,
    ano_publicacao SMALLINT CHECK (ano_publicacao BETWEEN 1000 AND 2200),
    id_editora INTEGER NOT NULL REFERENCES editora(id_editora) ON DELETE RESTRICT,
    id_categoria INTEGER NOT NULL REFERENCES categoria(id_categoria) ON DELETE RESTRICT,
    CONSTRAINT ck_livro_isbn_formato CHECK (
        regexp_replace(isbn, '[^0-9Xx]', '', 'g') ~ '^[0-9]{9}[0-9Xx]$'
        OR regexp_replace(isbn, '[^0-9]', '', 'g') ~ '^[0-9]{13}$'
    )
);

CREATE TABLE IF NOT EXISTS autor (
    id_autor SERIAL PRIMARY KEY,
    nome VARCHAR(180) NOT NULL,
    nacionalidade VARCHAR(100) NOT NULL DEFAULT 'Nao informada',
    CONSTRAINT uq_autor_nome_nacionalidade UNIQUE (nome, nacionalidade)
);

CREATE TABLE IF NOT EXISTS livro_autor (
    isbn VARCHAR(20) NOT NULL REFERENCES livro(isbn) ON DELETE CASCADE,
    id_autor INTEGER NOT NULL REFERENCES autor(id_autor) ON DELETE RESTRICT,
    ordem_autoria SMALLINT NOT NULL CHECK (ordem_autoria > 0),
    PRIMARY KEY (isbn, id_autor),
    CONSTRAINT uq_livro_autor_ordem UNIQUE (isbn, ordem_autoria)
);

CREATE TABLE IF NOT EXISTS exemplar (
    isbn VARCHAR(20) NOT NULL REFERENCES livro(isbn) ON DELETE CASCADE,
    numero_exemplar SMALLINT NOT NULL CHECK (numero_exemplar > 0),
    estado VARCHAR(20) NOT NULL DEFAULT 'BOM'
        CHECK (estado IN ('NOVO', 'BOM', 'REGULAR', 'RUIM', 'EM_REPARO')),
    localizacao VARCHAR(100) NOT NULL,
    PRIMARY KEY (isbn, numero_exemplar)
);

CREATE TABLE IF NOT EXISTS usuario (
    matricula INTEGER PRIMARY KEY,
    cpf VARCHAR(14) NOT NULL UNIQUE,
    nome VARCHAR(180) NOT NULL,
    email VARCHAR(254) NOT NULL UNIQUE,
    rua VARCHAR(160) NOT NULL,
    numero_endereco VARCHAR(20) NOT NULL,
    complemento VARCHAR(100),
    bairro VARCHAR(100) NOT NULL,
    cidade VARCHAR(100) NOT NULL,
    cep VARCHAR(10),
    ativo BOOLEAN NOT NULL DEFAULT TRUE,
    criado_em TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT ck_usuario_cpf_11_digitos CHECK (
        regexp_replace(cpf, '[^0-9]', '', 'g') ~ '^[0-9]{11}$'
    )
);

CREATE TABLE IF NOT EXISTS usuario_telefone (
    matricula INTEGER NOT NULL REFERENCES usuario(matricula) ON DELETE CASCADE,
    telefone VARCHAR(24) NOT NULL,
    tipo VARCHAR(20) NOT NULL DEFAULT 'CELULAR'
        CHECK (tipo IN ('CELULAR', 'RESIDENCIAL', 'COMERCIAL', 'OUTRO')),
    whatsapp BOOLEAN NOT NULL DEFAULT FALSE,
    PRIMARY KEY (matricula, telefone)
);

CREATE TABLE IF NOT EXISTS funcionario (
    id_funcionario SERIAL PRIMARY KEY,
    nome VARCHAR(180) NOT NULL,
    email VARCHAR(254) NOT NULL UNIQUE,
    ativo BOOLEAN NOT NULL DEFAULT TRUE,
    id_supervisor INTEGER REFERENCES funcionario(id_funcionario) ON DELETE SET NULL,
    CONSTRAINT ck_funcionario_nao_supervisiona_a_si CHECK (
        id_supervisor IS DISTINCT FROM id_funcionario
    )
);

CREATE TABLE IF NOT EXISTS emprestimo (
    id_emprestimo SERIAL PRIMARY KEY,
    matricula INTEGER NOT NULL REFERENCES usuario(matricula) ON DELETE RESTRICT,
    isbn VARCHAR(20) NOT NULL,
    numero_exemplar SMALLINT NOT NULL,
    id_funcionario_retirada INTEGER NOT NULL
        REFERENCES funcionario(id_funcionario) ON DELETE RESTRICT,
    id_funcionario_devolucao INTEGER
        REFERENCES funcionario(id_funcionario) ON DELETE RESTRICT,
    data_emprestimo DATE NOT NULL DEFAULT CURRENT_DATE,
    data_prevista DATE NOT NULL,
    data_devolucao DATE,
    multa_diaria NUMERIC(8,2) NOT NULL CHECK (multa_diaria >= 0),
    CONSTRAINT fk_emprestimo_exemplar FOREIGN KEY (isbn, numero_exemplar)
        REFERENCES exemplar(isbn, numero_exemplar) ON DELETE RESTRICT,
    CONSTRAINT ck_emprestimo_prazo CHECK (data_prevista >= data_emprestimo),
    CONSTRAINT ck_emprestimo_devolucao CHECK (
        data_devolucao IS NULL OR data_devolucao >= data_emprestimo
    ),
    CONSTRAINT ck_emprestimo_funcionario_devolucao CHECK (
        (data_devolucao IS NULL AND id_funcionario_devolucao IS NULL)
        OR (data_devolucao IS NOT NULL AND id_funcionario_devolucao IS NOT NULL)
    )
);

CREATE UNIQUE INDEX IF NOT EXISTS uq_exemplar_com_emprestimo_aberto
    ON emprestimo (isbn, numero_exemplar)
    WHERE data_devolucao IS NULL;

CREATE TABLE IF NOT EXISTS reserva (
    id_reserva SERIAL PRIMARY KEY,
    matricula INTEGER NOT NULL REFERENCES usuario(matricula) ON DELETE RESTRICT,
    isbn VARCHAR(20) NOT NULL REFERENCES livro(isbn) ON DELETE RESTRICT,
    data_reserva TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    situacao VARCHAR(12) NOT NULL DEFAULT 'ATIVA'
        CHECK (situacao IN ('ATIVA', 'ATENDIDA', 'CANCELADA')),
    atendida_em TIMESTAMPTZ,
    cancelada_em TIMESTAMPTZ,
    isbn_exemplar_atendido VARCHAR(20),
    numero_exemplar_atendido SMALLINT,
    id_funcionario_atendimento INTEGER
        REFERENCES funcionario(id_funcionario) ON DELETE RESTRICT,
    CONSTRAINT fk_reserva_exemplar_atendido
        FOREIGN KEY (isbn_exemplar_atendido, numero_exemplar_atendido)
        REFERENCES exemplar(isbn, numero_exemplar) ON DELETE RESTRICT
);

CREATE TABLE IF NOT EXISTS pagamento_multa (
    id_pagamento SERIAL PRIMARY KEY,
    id_emprestimo INTEGER NOT NULL REFERENCES emprestimo(id_emprestimo) ON DELETE RESTRICT,
    id_funcionario INTEGER NOT NULL REFERENCES funcionario(id_funcionario) ON DELETE RESTRICT,
    valor NUMERIC(8,2) NOT NULL CHECK (valor > 0),
    pago_em TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    forma_pagamento VARCHAR(16) NOT NULL DEFAULT 'DINHEIRO'
        CHECK (forma_pagamento IN ('DINHEIRO', 'PIX', 'CARTAO', 'TRANSFERENCIA'))
);

CREATE OR REPLACE VIEW v_emprestimos_abertos AS
SELECT
    emp.id_emprestimo, emp.matricula, u.nome AS usuario, emp.isbn, l.titulo,
    emp.data_prevista, GREATEST(CURRENT_DATE - emp.data_prevista, 0)::INTEGER AS dias_atraso
FROM emprestimo AS emp
JOIN usuario AS u ON u.matricula = emp.matricula
JOIN livro AS l ON l.isbn = emp.isbn
WHERE emp.data_devolucao IS NULL;

COMMIT;
"

BEGIN
CREATE TABLE
INSERT 0 1
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE INDEX
CREATE TABLE
CREATE TABLE
CREATE VIEW
COMMIT


In [3]:
%%bash
posql -c "
SELECT tablename as tabela_criada
FROM pg_catalog.pg_tables
WHERE schemaname = 'public'
ORDER BY tablename;
"

      tabela_criada      
-------------------------
 autor
 categoria
 configuracao_biblioteca
 editora
 emprestimo
 exemplar
 funcionario
 livro
 livro_autor
 pagamento_multa
 reserva
 usuario
 usuario_telefone
(13 rows)



---
# 6. Povoamento dos Dados de Teste

Nessa parte fazemos a inserção da massa de dados inicial ficticia.

In [4]:
%%bash
posql -c "
BEGIN;

INSERT INTO editora (id_editora, nome, cidade) VALUES
    (1, 'Companhia das Letras', 'Sao Paulo'),
    (2, 'HarperCollins Brasil', 'Rio de Janeiro'),
    (3, 'Rocco', 'Rio de Janeiro'),
    (4, 'Intrinseca', 'Rio de Janeiro'),
    (5, 'Suma de Letras', 'Sao Paulo')
ON CONFLICT (id_editora) DO UPDATE
SET nome = EXCLUDED.nome, cidade = EXCLUDED.cidade;

INSERT INTO categoria (id_categoria, nome, id_categoria_pai) VALUES
    (1, 'Ficcao e Literatura', NULL),
    (2, 'Literatura Brasileira', 1),
    (3, 'Fantasia', 1),
    (4, 'Ficcao Cientifica e Distopia', 1),
    (5, 'Suspense e Mistério', 1),
    (6, 'Nao-Ficcao', NULL),
    (7, 'Historia e Biografias', 6)
ON CONFLICT (id_categoria) DO UPDATE
SET nome = EXCLUDED.nome, id_categoria_pai = EXCLUDED.id_categoria_pai;

INSERT INTO autor (id_autor, nome, nacionalidade) VALUES
    (101, 'Machado de Assis', 'Brasileira'),
    (102, 'J.R.R. Tolkien', 'Britanica'),
    (103, 'George Orwell', 'Britanica'),
    (104, 'Agatha Christie', 'Britanica'),
    (105, 'Anne Frank', 'Alema'),
    (106, 'Yuval Noah Harari', 'Israelense'),
    (107, 'Ariano Suassuna', 'Brasileira'),
    (108, 'Stephen King', 'Estadunidense')
ON CONFLICT (id_autor) DO UPDATE
SET nome = EXCLUDED.nome, nacionalidade = EXCLUDED.nacionalidade;

INSERT INTO livro (isbn, titulo, ano_publicacao, id_editora, id_categoria) VALUES
    ('9788508041764', 'Dom Casmurro', 1899, 1, 2),
    ('9788533613379', 'O Senhor dos Aneis: A Sociedade do Anel', 1954, 2, 3),
    ('9788535914849', '1984', 1949, 1, 4),
    ('9788525409409', 'O Assassinato no Expresso do Oriente', 1934, 2, 5),
    ('9788501069509', 'O Diario de Anne Frank', 1947, 3, 7),
    ('9788535924299', 'Sapiens: Uma Breve Historia da Humanidade', 2011, 1, 7),
    ('9788520935514', 'O Auto da Compadecida', 1955, 3, 2),
    ('9788581050485', 'O Iluminado', 1977, 5, 5)
ON CONFLICT (isbn) DO UPDATE
SET titulo = EXCLUDED.titulo, ano_publicacao = EXCLUDED.ano_publicacao,
    id_editora = EXCLUDED.id_editora, id_categoria = EXCLUDED.id_categoria;

INSERT INTO livro_autor (isbn, id_autor, ordem_autoria) VALUES
    ('9788508041764', 101, 1),
    ('9788533613379', 102, 1),
    ('9788535914849', 103, 1),
    ('9788525409409', 104, 1),
    ('9788501069509', 105, 1),
    ('9788535924299', 106, 1),
    ('9788520935514', 107, 1),
    ('9788581050485', 108, 1)
ON CONFLICT (isbn, id_autor) DO UPDATE
SET ordem_autoria = EXCLUDED.ordem_autoria;

INSERT INTO exemplar (isbn, numero_exemplar, estado, localizacao) VALUES
    ('9788508041764', 1, 'BOM', 'Corredor A - Estante Literatura Brasileira'),
    ('9788508041764', 2, 'REGULAR', 'Corredor A - Estante Literatura Brasileira'),
    ('9788533613379', 1, 'BOM', 'Corredor B - Estante Fantasia'),
    ('9788533613379', 2, 'NOVO', 'Corredor B - Estante Fantasia'),
    ('9788535914849', 1, 'BOM', 'Corredor B - Estante Ficcao'),
    ('9788525409409', 1, 'BOM', 'Corredor C - Estante Suspense'),
    ('9788525409409', 2, 'REGULAR', 'Corredor C - Estante Suspense'),
    ('9788501069509', 1, 'RUIM', 'Corredor D - Estante Biografias'),
    ('9788501069509', 2, 'BOM', 'Corredor D - Estante Biografias'),
    ('9788535924299', 1, 'NOVO', 'Corredor D - Estante Historia'),
    ('9788535924299', 2, 'BOM', 'Corredor D - Estante Historia'),
    ('9788520935514', 1, 'BOM', 'Corredor A - Estante Literatura Brasileira'),
    ('9788581050485', 1, 'BOM', 'Corredor C - Estante Suspense')
ON CONFLICT (isbn, numero_exemplar) DO UPDATE
SET estado = EXCLUDED.estado, localizacao = EXCLUDED.localizacao;

INSERT INTO usuario
    (matricula, cpf, nome, email, rua, numero_endereco, complemento, bairro, cidade, cep)
VALUES
    (201, '09876543211', 'Mariana Medeiros de Queiroz', 'mariana.queiroz88@gmail.com',
     'Avenida Senador Salgado Filho', '1500', 'Apto 402, Bloco B', 'Lagoa Nova', 'Natal', '59056-000'),
    (202, '45612378900', 'Felipe Bezerra Cavalcanti', 'felipe.cavalcanti.rn@hotmail.com',
     'Rua Leonora Armstrong', '34', 'Casa com portao preto', 'Ponta Negra', 'Natal', '59092-230'),
    (203, '78945612311', 'Camila Soares Dantas', 'camilasdantas95@outlook.com',
     'Avenida Afonso Pena', '870', NULL, 'Tirol', 'Natal', '59020-100'),
    (204, '32165498722', 'Tiago da Silva Gadelha', 'tgadelha.dev@gmail.com',
     'Avenida Prudente de Morais', '4105', 'Ed. Tirol Way, Apto 1105', 'Lagoa Seca', 'Natal', '59022-545'),
    (205, '14725836933', 'Beatriz Nogueira Rocha', 'bia.nrocha@yahoo.com.br',
     'Rua dos Tororos', '215', 'Casa 2', 'Alecrim', 'Natal', '59030-010')
ON CONFLICT (matricula) DO UPDATE
SET cpf = EXCLUDED.cpf, nome = EXCLUDED.nome, email = EXCLUDED.email,
    rua = EXCLUDED.rua, numero_endereco = EXCLUDED.numero_endereco,
    complemento = EXCLUDED.complemento, bairro = EXCLUDED.bairro,
    cidade = EXCLUDED.cidade, cep = EXCLUDED.cep;

INSERT INTO usuario_telefone (matricula, telefone, tipo, whatsapp) VALUES
    (201, '(84) 99182-3341', 'CELULAR', TRUE),
    (201, '(84) 3211-5582', 'RESIDENCIAL', FALSE),
    (202, '(84) 98845-9102', 'CELULAR', TRUE),
    (203, '(84) 99402-7715', 'CELULAR', TRUE),
    (204, '(84) 99633-1289', 'CELULAR', TRUE),
    (205, '(84) 98755-4001', 'CELULAR', TRUE)
ON CONFLICT (matricula, telefone) DO UPDATE
SET tipo = EXCLUDED.tipo, whatsapp = EXCLUDED.whatsapp;

INSERT INTO funcionario (id_funcionario, nome, email, ativo, id_supervisor) VALUES
    (501, 'Isadora Marinho', 'isadora.marinho@biblioteca.gov.br', TRUE, NULL),
    (502, 'Rafael Cortez', 'rafael.cortez@biblioteca.gov.br', TRUE, 501),
    (503, 'Luciana Tavares', 'luciana.tavares@biblioteca.gov.br', TRUE, 501)
ON CONFLICT (id_funcionario) DO UPDATE
SET nome = EXCLUDED.nome, email = EXCLUDED.email,
    ativo = EXCLUDED.ativo, id_supervisor = EXCLUDED.id_supervisor;

INSERT INTO emprestimo
    (id_emprestimo, matricula, isbn, numero_exemplar, id_funcionario_retirada,
     id_funcionario_devolucao, data_emprestimo, data_prevista, data_devolucao, multa_diaria)
VALUES
    (1001, 201, '9788508041764', 1, 501, NULL,
     CURRENT_DATE - 20, CURRENT_DATE - 6, NULL, 1.00),
    (1002, 202, '9788533613379', 2, 502, NULL,
     CURRENT_DATE - 2, CURRENT_DATE + 12, NULL, 1.00),
    (1003, 203, '9788535914849', 1, 501, NULL,
     CURRENT_DATE - 4, CURRENT_DATE + 10, NULL, 1.00),
    (1004, 204, '9788501069509', 1, 503, 503,
     CURRENT_DATE - 20, CURRENT_DATE - 10, CURRENT_DATE - 3, 1.00),
    (1005, 205, '9788535924299', 1, 501, NULL,
     CURRENT_DATE - 17, CURRENT_DATE - 3, NULL, 1.00),
    (1006, 201, '9788581050485', 1, 502, NULL,
     CURRENT_DATE - 3, CURRENT_DATE + 11, NULL, 1.00),
    (1007, 202, '9788525409409', 1, 503, 503,
     CURRENT_DATE - 24, CURRENT_DATE - 10, CURRENT_DATE - 12, 1.00),
    (1008, 203, '9788508041764', 2, 502, 502,
     CURRENT_DATE - 30, CURRENT_DATE - 16, CURRENT_DATE - 2, 1.00)
ON CONFLICT (id_emprestimo) DO UPDATE
SET matricula = EXCLUDED.matricula, isbn = EXCLUDED.isbn,
    numero_exemplar = EXCLUDED.numero_exemplar,
    id_funcionario_retirada = EXCLUDED.id_funcionario_retirada,
    id_funcionario_devolucao = EXCLUDED.id_funcionario_devolucao,
    data_emprestimo = EXCLUDED.data_emprestimo,
    data_prevista = EXCLUDED.data_prevista,
    data_devolucao = EXCLUDED.data_devolucao,
    multa_diaria = EXCLUDED.multa_diaria;

INSERT INTO reserva
    (id_reserva, matricula, isbn, data_reserva, situacao)
VALUES
    (701, 203, '9788508041764', CURRENT_TIMESTAMP - INTERVAL '1 day', 'ATIVA'),
    (702, 204, '9788535924299', CURRENT_TIMESTAMP - INTERVAL '2 days', 'ATIVA')
ON CONFLICT (id_reserva) DO UPDATE
SET matricula = EXCLUDED.matricula, isbn = EXCLUDED.isbn,
    data_reserva = EXCLUDED.data_reserva, situacao = EXCLUDED.situacao,
    atendida_em = NULL, cancelada_em = NULL,
    isbn_exemplar_atendido = NULL, numero_exemplar_atendido = NULL,
    id_funcionario_atendimento = NULL;

INSERT INTO pagamento_multa
    (id_pagamento, id_emprestimo, id_funcionario, valor, pago_em, forma_pagamento)
VALUES
    (801, 1004, 503, 3.00, CURRENT_TIMESTAMP - INTERVAL '3 days', 'PIX')
ON CONFLICT (id_pagamento) DO UPDATE
SET id_emprestimo = EXCLUDED.id_emprestimo,
    id_funcionario = EXCLUDED.id_funcionario,
    valor = EXCLUDED.valor, pago_em = EXCLUDED.pago_em,
    forma_pagamento = EXCLUDED.forma_pagamento;

SELECT setval(pg_get_serial_sequence('editora', 'id_editora')::regclass, (SELECT MAX(id_editora) FROM editora), TRUE);
SELECT setval(pg_get_serial_sequence('categoria', 'id_categoria')::regclass, (SELECT MAX(id_categoria) FROM categoria), TRUE);
SELECT setval(pg_get_serial_sequence('autor', 'id_autor')::regclass, (SELECT MAX(id_autor) FROM autor), TRUE);
SELECT setval(pg_get_serial_sequence('funcionario', 'id_funcionario')::regclass, (SELECT MAX(id_funcionario) FROM funcionario), TRUE);
SELECT setval(pg_get_serial_sequence('emprestimo', 'id_emprestimo')::regclass, (SELECT MAX(id_emprestimo) FROM emprestimo), TRUE);
SELECT setval(pg_get_serial_sequence('reserva', 'id_reserva')::regclass, (SELECT MAX(id_reserva) FROM reserva), TRUE);
SELECT setval(pg_get_serial_sequence('pagamento_multa', 'id_pagamento')::regclass, (SELECT MAX(id_pagamento) FROM pagamento_multa), TRUE);

COMMIT;
"

BEGIN
INSERT 0 5
INSERT 0 7
INSERT 0 8
INSERT 0 8
INSERT 0 8
INSERT 0 13
INSERT 0 5
INSERT 0 6
INSERT 0 3
INSERT 0 8
INSERT 0 2
INSERT 0 1
 setval 
--------
      5
(1 row)

 setval 
--------
      7
(1 row)

 setval 
--------
    108
(1 row)

 setval 
--------
    503
(1 row)

 setval 
--------
   1008
(1 row)

 setval 
--------
    702
(1 row)

 setval 
--------
    801
(1 row)

COMMIT


In [5]:
%%bash
posql -c "SELECT isbn, titulo, ano_publicacao FROM livro LIMIT 5;"

     isbn      |                 titulo                  | ano_publicacao 
---------------+-----------------------------------------+----------------
 9788508041764 | Dom Casmurro                            |           1899
 9788533613379 | O Senhor dos Aneis: A Sociedade do Anel |           1954
 9788535914849 | 1984                                    |           1949
 9788525409409 | O Assassinato no Expresso do Oriente    |           1934
 9788501069509 | O Diario de Anne Frank                  |           1947
(5 rows)



---
# 7. Demonstração Prática de Uso
Testes de consultas relacioanis avançadas, verificação de disponibilidade física e cadastro em tempo real.

In [6]:
%%bash
posql -c "
SELECT l.isbn, l.titulo, c.nome as categoria, e.nome as editora
FROM livro l
JOIN categoria c ON l.id_categoria = c.id_categoria
JOIN editora e ON l.id_editora = e.id_editora
ORDER BY l.titulo
LIMIT 5;
"

     isbn      |                titulo                |          categoria           |       editora        
---------------+--------------------------------------+------------------------------+----------------------
 9788535914849 | 1984                                 | Ficcao Cientifica e Distopia | Companhia das Letras
 9788508041764 | Dom Casmurro                         | Literatura Brasileira        | Companhia das Letras
 9788525409409 | O Assassinato no Expresso do Oriente | Suspense e Mistério          | HarperCollins Brasil
 9788520935514 | O Auto da Compadecida                | Literatura Brasileira        | Rocco
 9788501069509 | O Diario de Anne Frank               | Historia e Biografias        | Rocco
(5 rows)



In [7]:
%%bash
posql -c "
SELECT e.isbn, l.titulo, e.numero_exemplar, e.estado, e.localizacao
FROM exemplar e
JOIN livro l ON e.isbn = l.isbn
ORDER BY l.titulo, e.numero_exemplar
LIMIT 5;
"

     isbn      |                titulo                | numero_exemplar | estado  |                localizacao                 
---------------+--------------------------------------+-----------------+---------+--------------------------------------------
 9788535914849 | 1984                                 |               1 | BOM     | Corredor B - Estante Ficcao
 9788508041764 | Dom Casmurro                         |               1 | BOM     | Corredor A - Estante Literatura Brasileira
 9788508041764 | Dom Casmurro                         |               2 | REGULAR | Corredor A - Estante Literatura Brasileira
 9788525409409 | O Assassinato no Expresso do Oriente |               1 | BOM     | Corredor C - Estante Suspense
 9788525409409 | O Assassinato no Expresso do Oriente |               2 | REGULAR | Corredor C - Estante Suspense
(5 rows)



In [8]:
%%bash
posql -c "
INSERT INTO usuario (matricula, cpf, nome, email, rua, numero_endereco, bairro, cidade)
VALUES (206, '99988877766', 'Alexandre Maciel Medeiros', 'alexandre.maciel.701@ufrn.edu.br', 'Av. Alguma Ai', '100', 'Tal', 'Natal');

SELECT matricula, cpf, nome, email FROM usuario ORDER BY matricula DESC LIMIT 3;
"

INSERT 0 1
 matricula |     cpf     |           nome            |              email               
-----------+-------------+---------------------------+----------------------------------
       206 | 99988877766 | Alexandre Maciel Medeiros | alexandre.maciel.701@ufrn.edu.br
       205 | 14725836933 | Beatriz Nogueira Rocha    | bia.nrocha@yahoo.com.br
       204 | 32165498722 | Tiago da Silva Gadelha    | tgadelha.dev@gmail.com
(3 rows)



## 7.1 Disponibilidade de Exemplares por Título




In [9]:
%%bash
posql -c "
SELECT l.titulo,
       COUNT(*) AS total_exemplares,
       COUNT(*) FILTER (WHERE emp.id_emprestimo IS NULL AND ex.estado <> 'EM_REPARO') AS disponiveis,
       COUNT(*) FILTER (WHERE emp.id_emprestimo IS NOT NULL) AS emprestados
FROM exemplar ex
JOIN livro l ON l.isbn = ex.isbn
LEFT JOIN emprestimo emp
       ON emp.isbn = ex.isbn AND emp.numero_exemplar = ex.numero_exemplar
      AND emp.data_devolucao IS NULL
GROUP BY l.titulo
ORDER BY l.titulo;
"

                  titulo                   | total_exemplares | disponiveis | emprestados 
-------------------------------------------+------------------+-------------+-------------
 1984                                      |                1 |           0 |           1
 Dom Casmurro                              |                2 |           1 |           1
 O Assassinato no Expresso do Oriente      |                2 |           2 |           0
 O Auto da Compadecida                     |                1 |           1 |           0
 O Diario de Anne Frank                    |                2 |           2 |           0
 O Iluminado                               |                1 |           0 |           1
 O Senhor dos Aneis: A Sociedade do Anel   |                2 |           1 |           1
 Sapiens: Uma Breve Historia da Humanidade |                2 |           1 |           1
(8 rows)



## 7.2 Cadastro de um Novo Título com Dois Autores e Três Exemplares




In [10]:
%%bash
echo '--- Livro e autores (CTE) ---'
posql -c "
WITH novo_livro AS (
    INSERT INTO livro (isbn, titulo, ano_publicacao, id_editora, id_categoria)
    VALUES ('9780000000019', 'Belas Maldicoes', 1990, 4, 3)
    RETURNING isbn
), autor_1 AS (
    INSERT INTO autor (nome, nacionalidade) VALUES ('Neil Gaiman', 'Britanica')
    RETURNING id_autor
), autor_2 AS (
    INSERT INTO autor (nome, nacionalidade) VALUES ('Terry Pratchett', 'Britanica')
    RETURNING id_autor
)
INSERT INTO livro_autor (isbn, id_autor, ordem_autoria)
SELECT nl.isbn, a.id_autor, a.ordem
FROM novo_livro nl
CROSS JOIN (
    SELECT id_autor, 1 AS ordem FROM autor_1
    UNION ALL
    SELECT id_autor, 2 FROM autor_2
) AS a;
"

echo '--- Três exemplares novos ---'
posql -c "
INSERT INTO exemplar (isbn, numero_exemplar, estado, localizacao)
SELECT '9780000000019', n, 'NOVO', 'Corredor B - Estante Fantasia'
FROM generate_series(1, 3) AS n;
"

echo '--- Conferindo o resultado ---'
posql -c "
SELECT l.isbn, l.titulo,
       string_agg(a.nome, ', ' ORDER BY la.ordem_autoria) AS autores,
       (SELECT COUNT(*) FROM exemplar ex WHERE ex.isbn = l.isbn) AS exemplares
FROM livro l
JOIN livro_autor la ON la.isbn = l.isbn
JOIN autor a ON a.id_autor = la.id_autor
WHERE l.isbn = '9780000000019'
GROUP BY l.isbn, l.titulo;
"

--- Livro e autores (CTE) ---
--- Três exemplares novos ---
INSERT 0 2
--- Conferindo o resultado ---
INSERT 0 3
     isbn      |     titulo      |           autores            | exemplares 
---------------+-----------------+------------------------------+------------
 9780000000019 | Belas Maldicoes | Neil Gaiman, Terry Pratchett |          3
(1 row)



## 7.3 Empréstimo de um Livro



In [11]:
%%bash
posql -c "
INSERT INTO emprestimo
    (matricula, isbn, numero_exemplar, id_funcionario_retirada,
     data_emprestimo, data_prevista, multa_diaria)
SELECT 202, ex.isbn, ex.numero_exemplar, 502,
       CURRENT_DATE, CURRENT_DATE + cfg.dias_emprestimo_padrao, cfg.multa_diaria
FROM exemplar ex
CROSS JOIN configuracao_biblioteca cfg
WHERE ex.isbn = '9788535924299'
  AND ex.estado <> 'EM_REPARO'
  AND NOT EXISTS (
    SELECT 1 FROM emprestimo e
    WHERE e.isbn = ex.isbn AND e.numero_exemplar = ex.numero_exemplar
      AND e.data_devolucao IS NULL
  )
ORDER BY ex.numero_exemplar
LIMIT 1
RETURNING id_emprestimo, matricula, isbn, numero_exemplar, data_prevista, multa_diaria;
"

 id_emprestimo | matricula |     isbn      | numero_exemplar | data_prevista | multa_diaria 
---------------+-----------+---------------+-----------------+---------------+--------------
          1009 |       202 | 9788535924299 |               2 | 2026-11-17    |         1.00
(1 row)

INSERT 0 1


## 7.4 Regras de Negócio que Barram um Empréstimo




In [12]:
%%bash
echo '--- a) Tentativa da Mariana (deve inserir 0 linhas) ---'
posql -c "
INSERT INTO emprestimo
    (matricula, isbn, numero_exemplar, id_funcionario_retirada,
     data_emprestimo, data_prevista, multa_diaria)
SELECT 201, ex.isbn, ex.numero_exemplar, 501,
       CURRENT_DATE, CURRENT_DATE + cfg.dias_emprestimo_padrao, cfg.multa_diaria
FROM exemplar ex
CROSS JOIN configuracao_biblioteca cfg
WHERE ex.isbn = '9788533613379'
  AND ex.estado <> 'EM_REPARO'
  AND NOT EXISTS (
    SELECT 1 FROM emprestimo e
    WHERE e.isbn = ex.isbn AND e.numero_exemplar = ex.numero_exemplar
      AND e.data_devolucao IS NULL
  )
  AND EXISTS (SELECT 1 FROM usuario u WHERE u.matricula = 201 AND u.ativo)
  AND NOT EXISTS (
      SELECT 1 FROM emprestimo a
      WHERE a.matricula = 201 AND a.data_devolucao IS NULL AND a.data_prevista < CURRENT_DATE
  )
ORDER BY ex.numero_exemplar
LIMIT 1
RETURNING id_emprestimo;
"

echo '--- Motivo: empréstimos em atraso da Mariana ---'
posql -c "
SELECT id_emprestimo, titulo, data_prevista, dias_atraso
FROM v_emprestimos_abertos
WHERE matricula = 201 AND dias_atraso > 0;
"

echo '--- b) Forçando o empréstimo do exemplar 1 de Sapiens (deve dar ERRO) ---'
posql -c "
INSERT INTO emprestimo
    (matricula, isbn, numero_exemplar, id_funcionario_retirada, data_prevista, multa_diaria)
VALUES (203, '9788535924299', 1, 501, CURRENT_DATE + 14, 1.00);
"

--- a) Tentativa da Mariana (deve inserir 0 linhas) ---
--- Motivo: empréstimos em atraso da Mariana ---
 id_emprestimo 
---------------
(0 rows)

INSERT 0 0
--- b) Forçando o empréstimo do exemplar 1 de Sapiens (deve dar ERRO) ---
 id_emprestimo |    titulo    | data_prevista | dias_atraso 
---------------+--------------+---------------+-------------
          1001 | Dom Casmurro | 2026-10-28    |           6
(1 row)



ERROR:  duplicate key value violates unique constraint "uq_exemplar_com_emprestimo_aberto"
DETAIL:  Key (isbn, numero_exemplar)=(9788535924299, 1) already exists.


## 7.5 Painel de Empréstimos em Aberto




In [13]:
%%bash
posql -c "
SELECT id_emprestimo, matricula, usuario, titulo, data_prevista, dias_atraso
FROM v_emprestimos_abertos
ORDER BY dias_atraso DESC, id_emprestimo;
"

 id_emprestimo | matricula |           usuario           |                  titulo                   | data_prevista | dias_atraso 
---------------+-----------+-----------------------------+-------------------------------------------+---------------+-------------
          1001 |       201 | Mariana Medeiros de Queiroz | Dom Casmurro                              | 2026-10-28    |           6
          1005 |       205 | Beatriz Nogueira Rocha      | Sapiens: Uma Breve Historia da Humanidade | 2026-10-31    |           3
          1002 |       202 | Felipe Bezerra Cavalcanti   | O Senhor dos Aneis: A Sociedade do Anel   | 2026-11-15    |           0
          1003 |       203 | Camila Soares Dantas        | 1984                                      | 2026-11-13    |           0
          1006 |       201 | Mariana Medeiros de Queiroz | O Iluminado                               | 2026-11-14    |           0
          1009 |       202 | Felipe Bezerra Cavalcanti   | Sapiens: Uma Breve His

## 7.6 Fila de Reservas FIFO




In [14]:
%%bash
echo '--- Reserva da Mariana para Sapiens ---'
posql -c "
INSERT INTO reserva (matricula, isbn)
SELECT 201, '9788535924299'
WHERE NOT EXISTS (
        SELECT 1 FROM exemplar ex
        WHERE ex.isbn = '9788535924299' AND ex.estado <> 'EM_REPARO' AND NOT EXISTS (
    SELECT 1 FROM emprestimo e
    WHERE e.isbn = ex.isbn AND e.numero_exemplar = ex.numero_exemplar
      AND e.data_devolucao IS NULL
  )
      )
  AND NOT EXISTS (
        SELECT 1 FROM reserva r
        WHERE r.matricula = 201 AND r.isbn = '9788535924299' AND r.situacao = 'ATIVA'
      )
RETURNING id_reserva, matricula, isbn, situacao;
"

echo '--- Fila de espera atual ---'
posql -c "
SELECT l.titulo,
       ROW_NUMBER() OVER (PARTITION BY r.isbn ORDER BY r.data_reserva, r.id_reserva) AS posicao,
       u.nome AS usuario, r.id_reserva, r.situacao
FROM reserva r
JOIN livro l ON l.isbn = r.isbn
JOIN usuario u ON u.matricula = r.matricula
WHERE r.situacao = 'ATIVA'
ORDER BY l.titulo, posicao;
"

--- Reserva da Mariana para Sapiens ---
--- Fila de espera atual ---
 id_reserva | matricula |     isbn      | situacao 
------------+-----------+---------------+----------
        703 |       201 | 9788535924299 | ATIVA
(1 row)

INSERT 0 1
                  titulo                   | posicao |           usuario           | id_reserva | situacao 
-------------------------------------------+---------+-----------------------------+------------+----------
 Dom Casmurro                              |       1 | Camila Soares Dantas        |        701 | ATIVA
 Sapiens: Uma Breve Historia da Humanidade |       1 | Tiago da Silva Gadelha      |        702 | ATIVA
 Sapiens: Uma Breve Historia da Humanidade |       2 | Mariana Medeiros de Queiroz |        703 | ATIVA
(3 rows)



## 7.7 Devolução de um Livro




In [15]:
%%bash
posql -c "
UPDATE emprestimo
SET data_devolucao = CURRENT_DATE,
    id_funcionario_devolucao = 502
WHERE id_emprestimo = 1005
  AND data_devolucao IS NULL
RETURNING id_emprestimo, matricula, isbn, numero_exemplar,
          data_prevista, data_devolucao,
          GREATEST(data_devolucao - data_prevista, 0) AS dias_atraso,
          GREATEST(data_devolucao - data_prevista, 0) * multa_diaria AS multa_devida;
"

 id_emprestimo | matricula |     isbn      | numero_exemplar | data_prevista | data_devolucao | dias_atraso | multa_devida 
---------------+-----------+---------------+-----------------+---------------+----------------+-------------+--------------
          1005 |       205 | 9788535924299 |               1 | 2026-10-31    | 2026-11-03     |           3 |         3.00
(1 row)

UPDATE 1


## 7.8 Atendimento da Reserva: Primeiro da Fila Leva o Exemplar




In [16]:
%%bash
echo '--- Atendimento + empréstimo (atômico) ---'
posql -c "
WITH proxima AS (
    SELECT id_reserva, matricula, isbn
    FROM reserva
    WHERE isbn = '9788535924299' AND situacao = 'ATIVA'
    ORDER BY data_reserva, id_reserva
    LIMIT 1
    FOR UPDATE
), atendida AS (
    UPDATE reserva r
    SET situacao = 'ATENDIDA',
        atendida_em = CURRENT_TIMESTAMP,
        isbn_exemplar_atendido = '9788535924299',
        numero_exemplar_atendido = 1,
        id_funcionario_atendimento = 502
    FROM proxima p
    WHERE r.id_reserva = p.id_reserva
    RETURNING r.id_reserva, r.matricula, r.isbn
)
INSERT INTO emprestimo
    (matricula, isbn, numero_exemplar, id_funcionario_retirada,
     data_emprestimo, data_prevista, multa_diaria)
SELECT a.matricula, a.isbn, 1, 502,
       CURRENT_DATE, CURRENT_DATE + cfg.dias_emprestimo_padrao, cfg.multa_diaria
FROM atendida a
CROSS JOIN configuracao_biblioteca cfg
RETURNING id_emprestimo, matricula, isbn, numero_exemplar, data_prevista;
"

echo '--- Situação das reservas de Sapiens ---'
posql -c "
SELECT r.id_reserva, u.nome AS usuario, r.situacao, r.atendida_em IS NOT NULL AS foi_atendida,
       r.numero_exemplar_atendido AS exemplar, r.id_funcionario_atendimento AS func
FROM reserva r JOIN usuario u ON u.matricula = r.matricula
WHERE r.isbn = '9788535924299'
ORDER BY r.data_reserva, r.id_reserva;
"

--- Atendimento + empréstimo (atômico) ---
--- Situação das reservas de Sapiens ---
 id_emprestimo | matricula |     isbn      | numero_exemplar | data_prevista 
---------------+-----------+---------------+-----------------+---------------
          1011 |       204 | 9788535924299 |               1 | 2026-11-17
(1 row)

INSERT 0 1
 id_reserva |           usuario           | situacao | foi_atendida | exemplar | func 
------------+-----------------------------+----------+--------------+----------+------
        702 | Tiago da Silva Gadelha      | ATENDIDA | t            |        1 |  502
        703 | Mariana Medeiros de Queiroz | ATIVA    | f            |          |     
(2 rows)



## 7.9 Cancelamento de Reserva



In [17]:
%%bash
posql -c "
UPDATE reserva
SET situacao = 'CANCELADA', cancelada_em = CURRENT_TIMESTAMP
WHERE matricula = 201 AND isbn = '9788535924299' AND situacao = 'ATIVA'
RETURNING id_reserva, matricula, situacao, cancelada_em;
"

echo '--- Fila de Sapiens depois do cancelamento (deve estar vazia) ---'
posql -c "
SELECT COUNT(*) AS reservas_ativas_sapiens
FROM reserva
WHERE isbn = '9788535924299' AND situacao = 'ATIVA';
"

--- Fila de Sapiens depois do cancelamento (deve estar vazia) ---
 id_reserva | matricula | situacao  |         cancelada_em          
------------+-----------+-----------+-------------------------------
        703 |       201 | CANCELADA | 2026-11-03 03:18:42.632113+00
(1 row)

UPDATE 1
 reservas_ativas_sapiens 
-------------------------
                       0
(1 row)



## 7.10 Devolução em Atraso e Multa



In [18]:
%%bash
posql -c "
UPDATE emprestimo
SET data_devolucao = CURRENT_DATE,
    id_funcionario_devolucao = 503
WHERE id_emprestimo = 1001
  AND data_devolucao IS NULL
RETURNING id_emprestimo, matricula, data_prevista, data_devolucao,
          GREATEST(data_devolucao - data_prevista, 0) AS dias_atraso,
          GREATEST(data_devolucao - data_prevista, 0) * multa_diaria AS multa_devida;
"

 id_emprestimo | matricula | data_prevista | data_devolucao | dias_atraso | multa_devida 
---------------+-----------+---------------+----------------+-------------+--------------
          1001 |       201 | 2026-10-28    | 2026-11-03     |           6 |         6.00
(1 row)

UPDATE 1


## 7.11 Visão de Saldo de Multas



In [19]:
%%bash
echo '--- Criação da visão ---'
posql -c "
CREATE OR REPLACE VIEW v_saldo_multas AS
SELECT t.id_emprestimo, t.matricula, u.nome AS usuario, l.titulo,
       t.data_prevista, t.data_devolucao, t.dias_atraso, t.multa_total,
       COALESCE(pg.total_pago, 0) AS total_pago,
       t.multa_total - COALESCE(pg.total_pago, 0) AS saldo_devedor
FROM (
    SELECT emp.*,
           GREATEST(COALESCE(emp.data_devolucao, CURRENT_DATE) - emp.data_prevista, 0)::INTEGER AS dias_atraso,
           (GREATEST(COALESCE(emp.data_devolucao, CURRENT_DATE) - emp.data_prevista, 0)
            * emp.multa_diaria)::NUMERIC(10,2) AS multa_total
    FROM emprestimo emp
) AS t
JOIN usuario u ON u.matricula = t.matricula
JOIN livro l ON l.isbn = t.isbn
LEFT JOIN (
    SELECT id_emprestimo, SUM(valor) AS total_pago
    FROM pagamento_multa
    GROUP BY id_emprestimo
) AS pg ON pg.id_emprestimo = t.id_emprestimo;
"

echo '--- Empréstimos com multa ---'
posql -c "
SELECT id_emprestimo, usuario, titulo, dias_atraso, multa_total, total_pago, saldo_devedor
FROM v_saldo_multas
WHERE multa_total > 0
ORDER BY id_emprestimo;
"

--- Criação da visão ---
--- Empréstimos com multa ---
CREATE VIEW


ERROR:  relation "v_saldo_multas" does not exist
LINE 3: FROM v_saldo_multas
             ^


## 7.12 Pagamento de Multa



In [20]:
%%bash
echo '--- 1) Pagamento parcial de R$ 4,00 (PIX) ---'
posql -c "
INSERT INTO pagamento_multa (id_emprestimo, id_funcionario, valor, forma_pagamento)
SELECT id_emprestimo, 503, 4.00, 'PIX'
FROM v_saldo_multas
WHERE id_emprestimo = 1001 AND saldo_devedor >= 4.00
RETURNING id_pagamento, id_emprestimo, valor, forma_pagamento;
"

echo '--- 2) Tentativa de pagar R$ 5,00 com saldo menor (deve inserir 0 linhas) ---'
posql -c "
INSERT INTO pagamento_multa (id_emprestimo, id_funcionario, valor, forma_pagamento)
SELECT id_emprestimo, 503, 5.00, 'DINHEIRO'
FROM v_saldo_multas
WHERE id_emprestimo = 1001 AND saldo_devedor >= 5.00
RETURNING id_pagamento;
"

echo '--- 3) Quitação do saldo restante em dinheiro ---'
posql -c "
INSERT INTO pagamento_multa (id_emprestimo, id_funcionario, valor, forma_pagamento)
SELECT id_emprestimo, 503, saldo_devedor, 'DINHEIRO'
FROM v_saldo_multas
WHERE id_emprestimo = 1001 AND saldo_devedor > 0
RETURNING id_pagamento, id_emprestimo, valor, forma_pagamento;
"

echo '--- 4) Extrato final do empréstimo 1001 ---'
posql -c "
SELECT id_emprestimo, usuario, multa_total, total_pago, saldo_devedor
FROM v_saldo_multas
WHERE id_emprestimo = 1001;
"

--- 1) Pagamento parcial de R$ 4,00 (PIX) ---
--- 2) Tentativa de pagar R$ 5,00 com saldo menor (deve inserir 0 linhas) ---
 id_pagamento | id_emprestimo | valor | forma_pagamento 
--------------+---------------+-------+-----------------
          802 |          1001 |  4.00 | PIX
(1 row)

INSERT 0 1
 id_pagamento 
--------------
(0 rows)

INSERT 0 0
--- 3) Quitação do saldo restante em dinheiro ---
--- 4) Extrato final do empréstimo 1001 ---
 id_pagamento | id_emprestimo | valor | forma_pagamento 
--------------+---------------+-------+-----------------
          803 |          1001 |  2.00 | DINHEIRO
(1 row)

INSERT 0 1
 id_emprestimo |           usuario           | multa_total | total_pago | saldo_devedor 
---------------+-----------------------------+-------------+------------+---------------
          1001 | Mariana Medeiros de Queiroz |        6.00 |       6.00 |          0.00
(1 row)



## 7.13 Transação com `ROLLBACK`: Simular sem Efetivar




In [21]:
%%bash
cat > /tmp/simulacao_emprestimo.sql <<'EOF'
BEGIN;

INSERT INTO emprestimo
    (matricula, isbn, numero_exemplar, id_funcionario_retirada,
     data_emprestimo, data_prevista, multa_diaria)
VALUES (203, '9788533613379', 1, 501, CURRENT_DATE, CURRENT_DATE + 14, 1.00)
RETURNING id_emprestimo, matricula, isbn, numero_exemplar;

SELECT 'dentro da transacao' AS momento, COUNT(*) AS emprestimos_abertos
FROM emprestimo WHERE data_devolucao IS NULL;

ROLLBACK;

SELECT 'apos o rollback' AS momento, COUNT(*) AS emprestimos_abertos
FROM emprestimo WHERE data_devolucao IS NULL;
EOF
posql -f /tmp/simulacao_emprestimo.sql

BEGIN
 id_emprestimo | matricula |     isbn      | numero_exemplar 
---------------+-----------+---------------+-----------------
          1012 |       203 | 9788533613379 |               1
(1 row)

INSERT 0 1
       momento       | emprestimos_abertos 
---------------------+---------------------
 dentro da transacao |                   6
(1 row)

ROLLBACK
     momento     | emprestimos_abertos 
-----------------+---------------------
 apos o rollback |                   5
(1 row)



## 7.14 Relatórios Gerenciais



In [22]:
%%bash
echo '--- a) Livros mais emprestados ---'
posql -c "
SELECT l.titulo, COUNT(*) AS total_emprestimos
FROM emprestimo e
JOIN livro l ON l.isbn = e.isbn
GROUP BY l.titulo
ORDER BY total_emprestimos DESC, l.titulo
LIMIT 5;
"

echo '--- b) Livros que nunca foram emprestados ---'
posql -c "
SELECT l.isbn, l.titulo
FROM livro l
WHERE NOT EXISTS (SELECT 1 FROM emprestimo e WHERE e.isbn = l.isbn)
ORDER BY l.titulo;
"

echo '--- c) Ranking de devedores (saldo de multas em aberto por usuário) ---'
posql -c "
SELECT matricula, usuario, SUM(saldo_devedor) AS total_devido
FROM v_saldo_multas
GROUP BY matricula, usuario
HAVING SUM(saldo_devedor) > 0
ORDER BY total_devido DESC;
"

echo '--- d) Arrecadação por forma de pagamento ---'
posql -c "
SELECT forma_pagamento, COUNT(*) AS pagamentos, SUM(valor) AS total_arrecadado
FROM pagamento_multa
GROUP BY forma_pagamento
ORDER BY total_arrecadado DESC;
"

echo '--- e) Desempenho da equipe (com o supervisor de cada funcionário) ---'
posql -c "
SELECT f.nome AS funcionario,
       COALESCE(s.nome, '-') AS supervisor,
       (SELECT COUNT(*) FROM emprestimo e WHERE e.id_funcionario_retirada = f.id_funcionario) AS entregas,
       (SELECT COUNT(*) FROM emprestimo e WHERE e.id_funcionario_devolucao = f.id_funcionario) AS devolucoes,
       (SELECT COALESCE(SUM(p.valor), 0) FROM pagamento_multa p WHERE p.id_funcionario = f.id_funcionario) AS multas_recebidas
FROM funcionario f
LEFT JOIN funcionario s ON s.id_funcionario = f.id_supervisor
ORDER BY f.id_funcionario;
"

echo '--- f) Histórico completo de um leitor (Mariana, matrícula 201) ---'
posql -c "
SELECT emp.id_emprestimo, l.titulo, emp.numero_exemplar AS ex,
       emp.data_emprestimo, emp.data_prevista, emp.data_devolucao,
       CASE WHEN emp.data_devolucao IS NULL THEN 'EM ABERTO' ELSE 'DEVOLVIDO' END AS situacao,
       fr.nome AS entregue_por, fd.nome AS recebido_por
FROM emprestimo emp
JOIN livro l ON l.isbn = emp.isbn
JOIN funcionario fr ON fr.id_funcionario = emp.id_funcionario_retirada
LEFT JOIN funcionario fd ON fd.id_funcionario = emp.id_funcionario_devolucao
WHERE emp.matricula = 201
ORDER BY emp.data_emprestimo;
"

--- a) Livros mais emprestados ---
--- b) Livros que nunca foram emprestados ---
                  titulo                   | total_emprestimos 
-------------------------------------------+-------------------
 Sapiens: Uma Breve Historia da Humanidade |                 3
 Dom Casmurro                              |                 2
 1984                                      |                 1
 O Assassinato no Expresso do Oriente      |                 1
 O Diario de Anne Frank                    |                 1
(5 rows)

--- c) Ranking de devedores (saldo de multas em aberto por usuário) ---
     isbn      |        titulo         
---------------+-----------------------
 9780000000019 | Belas Maldicoes
 9788520935514 | O Auto da Compadecida
(2 rows)

--- d) Arrecadação por forma de pagamento ---
 matricula |        usuario         | total_devido 
-----------+------------------------+--------------
       203 | Camila Soares Dantas   |        14.00
       204 | Tiago da Silva Gad

### 7.15 Hierarquia de Categorias com Consulta Recursiva



In [23]:
%%bash
echo '--- Árvore com nível e caminho ---'
posql -c "
WITH RECURSIVE arvore AS (
    SELECT id_categoria, nome, 0 AS nivel, nome::TEXT AS caminho
    FROM categoria
    WHERE id_categoria_pai IS NULL
    UNION ALL
    SELECT c.id_categoria, c.nome, a.nivel + 1, a.caminho || ' > ' || c.nome
    FROM categoria c
    JOIN arvore a ON c.id_categoria_pai = a.id_categoria
)
SELECT repeat('   ', nivel) || nome AS categoria, caminho,
       (SELECT COUNT(*) FROM livro l WHERE l.id_categoria = arvore.id_categoria) AS livros_diretos
FROM arvore
ORDER BY caminho;
"

echo '--- Livros por categoria raiz (somando as subcategorias) ---'
posql -c "
WITH RECURSIVE sub AS (
    SELECT id_categoria, id_categoria AS id_raiz, nome AS raiz
    FROM categoria
    WHERE id_categoria_pai IS NULL
    UNION ALL
    SELECT c.id_categoria, s.id_raiz, s.raiz
    FROM categoria c
    JOIN sub s ON c.id_categoria_pai = s.id_categoria
)
SELECT s.raiz AS categoria_raiz, COUNT(l.isbn) AS livros
FROM sub s
LEFT JOIN livro l ON l.id_categoria = s.id_categoria
GROUP BY s.raiz
ORDER BY livros DESC;
"

--- Árvore com nível e caminho ---
--- Livros por categoria raiz (somando as subcategorias) ---
            categoria            |                      caminho                       | livros_diretos 
---------------------------------+----------------------------------------------------+----------------
 Ficcao e Literatura             | Ficcao e Literatura                                |              0
    Fantasia                     | Ficcao e Literatura > Fantasia                     |              2
    Ficcao Cientifica e Distopia | Ficcao e Literatura > Ficcao Cientifica e Distopia |              1
    Literatura Brasileira        | Ficcao e Literatura > Literatura Brasileira        |              2
    Suspense e Mistério          | Ficcao e Literatura > Suspense e Mistério          |              2
 Nao-Ficcao                      | Nao-Ficcao                                         |              0
    Historia e Biografias        | Nao-Ficcao > Historia e Biografias         

## 7.16 Testes de Integridade: o Banco se Defendendo

Cada comando abaixo **deve falhar**, e a mensagem de erro mostra qual regra do esquema atuou.

In [24]:
%%bash
echo '--- 1) CPF com menos de 11 dígitos (ck_usuario_cpf_11_digitos) ---'
posql -c "
INSERT INTO usuario (matricula, cpf, nome, email, rua, numero_endereco, bairro, cidade)
VALUES (207, '123', 'Teste CPF', 'teste.cpf@exemplo.com', 'Rua A', '1', 'Centro', 'Natal');
"

echo '--- 2) CPF duplicado (UNIQUE) ---'
posql -c "
INSERT INTO usuario (matricula, cpf, nome, email, rua, numero_endereco, bairro, cidade)
VALUES (207, '09876543211', 'Teste Duplicado', 'teste.dup@exemplo.com', 'Rua A', '1', 'Centro', 'Natal');
"

echo '--- 3) ISBN em formato inválido (ck_livro_isbn_formato) ---'
posql -c "
INSERT INTO livro (isbn, titulo, ano_publicacao, id_editora, id_categoria)
VALUES ('ABC-123', 'Livro Invalido', 2020, 1, 1);
"

echo '--- 4) Estado de conservação inexistente (CHECK em exemplar.estado) ---'
posql -c "
INSERT INTO exemplar (isbn, numero_exemplar, estado, localizacao)
VALUES ('9788508041764', 9, 'DESTRUIDO', 'Corredor A');
"

echo '--- 5) Prazo anterior à data do empréstimo (ck_emprestimo_prazo) ---'
posql -c "
INSERT INTO emprestimo (matricula, isbn, numero_exemplar, id_funcionario_retirada,
                        data_emprestimo, data_prevista, multa_diaria)
VALUES (203, '9788533613379', 1, 501, CURRENT_DATE, CURRENT_DATE - 1, 1.00);
"

echo '--- 6) Devolução sem informar o funcionário (ck_emprestimo_funcionario_devolucao) ---'
posql -c "
UPDATE emprestimo SET data_devolucao = CURRENT_DATE WHERE id_emprestimo = 1002;
"

echo '--- 7) Apagar um livro que ainda é referenciado por reservas e empréstimos (ON DELETE RESTRICT) ---'
posql -c "
DELETE FROM livro WHERE isbn = '9788508041764';
"

echo '--- 8) Apagar um usuário com empréstimos (ON DELETE RESTRICT) ---'
posql -c "
DELETE FROM usuario WHERE matricula = 202;
"

echo '--- 9) Categoria pai de si mesma (ck_categoria_nao_pode_ser_pai_de_si) ---'
posql -c "
UPDATE categoria SET id_categoria_pai = 3 WHERE id_categoria = 3;
"

--- 1) CPF com menos de 11 dígitos (ck_usuario_cpf_11_digitos) ---
--- 2) CPF duplicado (UNIQUE) ---
--- 3) ISBN em formato inválido (ck_livro_isbn_formato) ---
--- 4) Estado de conservação inexistente (CHECK em exemplar.estado) ---
--- 5) Prazo anterior à data do empréstimo (ck_emprestimo_prazo) ---
--- 6) Devolução sem informar o funcionário (ck_emprestimo_funcionario_devolucao) ---
--- 7) Apagar um livro que ainda é referenciado por reservas e empréstimos (ON DELETE RESTRICT) ---
--- 8) Apagar um usuário com empréstimos (ON DELETE RESTRICT) ---
--- 9) Categoria pai de si mesma (ck_categoria_nao_pode_ser_pai_de_si) ---


ERROR:  new row for relation "usuario" violates check constraint "ck_usuario_cpf_11_digitos"
DETAIL:  Failing row contains (207, 123, Teste CPF, teste.cpf@exemplo.com, Rua A, 1, null, Centro, Natal, null, t, 2026-11-03 03:18:45.027938+00).
ERROR:  duplicate key value violates unique constraint "usuario_cpf_key"
DETAIL:  Key (cpf)=(09876543211) already exists.
ERROR:  new row for relation "livro" violates check constraint "ck_livro_isbn_formato"
DETAIL:  Failing row contains (ABC-123, Livro Invalido, 2020, 1, 1).
ERROR:  new row for relation "exemplar" violates check constraint "exemplar_estado_check"
DETAIL:  Failing row contains (9788508041764, 9, DESTRUIDO, Corredor A).
ERROR:  new row for relation "emprestimo" violates check constraint "ck_emprestimo_prazo"
DETAIL:  Failing row contains (1013, 203, 9788533613379, 1, 501, null, 2026-11-03, 2026-11-02, null, 1.00).
ERROR:  new row for relation "emprestimo" violates check constraint "ck_emprestimo_funcionario_devolucao"
DETAIL:  Failin

### Alternativa ao `DELETE` de usuário: inativação (exclusão lógica)

Como o histórico de empréstimos deve ser preservado, o caminho correto é marcar o usuário como inativo, e o `INSERT ... SELECT` de empréstimo (seção 7.4) já recusa usuários com `ativo = FALSE`.

In [25]:
%%bash
posql -c "
UPDATE usuario SET ativo = FALSE WHERE matricula = 205
RETURNING matricula, nome, ativo;
"

 matricula |          nome          | ativo 
-----------+------------------------+-------
       205 | Beatriz Nogueira Rocha | f
(1 row)

UPDATE 1


## 7.17 Reset: Voltar ao Estado Inicial



In [26]:
%%bash
posql -c "
BEGIN;
DELETE FROM pagamento_multa WHERE id_pagamento > 801;
DELETE FROM emprestimo WHERE id_emprestimo > 1008;
DELETE FROM reserva WHERE id_reserva > 702;
DELETE FROM livro_autor WHERE isbn = '9780000000019';
DELETE FROM exemplar WHERE isbn = '9780000000019';
DELETE FROM livro WHERE isbn = '9780000000019';
DELETE FROM autor WHERE nome IN ('Neil Gaiman', 'Terry Pratchett');
UPDATE usuario SET ativo = TRUE WHERE matricula = 205;
COMMIT;
DROP VIEW IF EXISTS v_saldo_multas;
"

BEGIN
DELETE 2
DELETE 2
DELETE 1
DELETE 2
DELETE 3
DELETE 1
DELETE 2
UPDATE 1
COMMIT
DROP VIEW


---
# 8. Backup
 A célula abaixo usa o `pg_dump` para exportar a estrutura e os dados do banco para um arquivo físico chamado `backup_banco_biblioteca.sql`. O arquivo estará disponível imediatamente na aba de **Arquivos** à esquerda do Colab para download para uma possível visualização em uma plataforma de gerenciamento de banco de dados SQL.

In [27]:
!sudo -u postgres pg_dump biblioteca > backup_banco_biblioteca.sql
print("Backup salvo em backup_banco_biblioteca.sql")

Backup salvo em backup_banco_biblioteca.sql
